In [73]:
# -*- encoding:utf-8 -*-
from __future__ import print_function
import os
import json
import time
import sys
#from similarity import similarity_main
from similarity import gensim_vec
import gensim
import numpy as np
import distance
import jieba.posseg as pesg
import sklearn
from sklearn.feature_extraction.text import CountVectorizer

In [74]:
def load_danmu(file_path):
    datas = []
    lines = open(file_path, 'r', encoding='utf8').read().strip().split('\n')
    for line in lines:
        data = json.loads(line)
        datas.append(data)
    return datas

In [75]:
# 1. 加载词向量
wv_from_text = gensim_vec()
#wv_from_text=[]

In [76]:
def save_similarity_cosine(similarity_list, matrix_similarity_dir):
    print("---save_similarity_cosine----------")
    wr = open(matrix_similarity_dir, 'w', encoding='utf8')
    wr.write(str(similarity_list))
    wr.close()

In [77]:
def dump_to_json(datas, fout):
    for data in datas:
        fout.write(json.dumps(data, sort_keys=True, separators=(',', ': '), ensure_ascii=False))
        fout.write('\n')
    fout.close()

In [99]:
def jaccard_similarity(s1, s2):
    def add_space(s):
        return ' '.join(list(s))
    # 将字中间加入空格
    s1, s2 = add_space(s1), add_space(s2)
    # 转化为TF矩阵
    cv = CountVectorizer(tokenizer=lambda s: s.split())
    corpus = [s1, s2]
    vectors = cv.fit_transform(corpus).toarray()
    # 求交集
    numerator = np.sum(np.min(vectors, axis=0))
    # 求并集
    denominator = np.sum(np.max(vectors, axis=0))
    # 计算杰卡德系数
    return  numerator**2 /denominator

In [100]:
# 计算word的ngrams词组
def compute_ngrams(word, min_n, max_n):
    # BOW, EOW = ('<', '>')  # Used by FastText to attach to all words as prefix and suffix
    extended_word = word
    ngrams = []
    for ngram_length in range(min_n, min(len(extended_word), max_n) + 1):
        for i in range(0, len(extended_word) - ngram_length + 1):
            ngrams.append(extended_word[i:i + ngram_length])
    return list(set(ngrams))

In [101]:
# 不在词典的情况下,计算相近向量
def word_vec(word, wv_from_text, min_n=1, max_n=3):
    # 确认词向量维度
    word_size = wv_from_text.wv.syn0[0].shape[0]
    # 计算word的ngrams词组
    ngrams = compute_ngrams(word, min_n=min_n, max_n=max_n)
    # 如果在词典之中，直接返回词向量
    if word in wv_from_text.index2word:
        return wv_from_text[word]
    else:
        # 不在词典的情况下，计算与词相近的词向量
        word_vec = np.zeros(word_size, dtype=np.float32)
        ngrams_found = 0
        ngrams_single = [ng for ng in ngrams if len(ng) == 1] 
        ngrams_more = [ng for ng in ngrams if len(ng) > 1]  
        # 先只接受2个单词长度以上的词向量
        for ngram in ngrams_more:
            if ngram in wv_from_text.index2word:
                word_vec += wv_from_text[ngram]
                ngrams_found += 1
                # print(ngram)
        # 如果，没有匹配到，那么最后是考虑单个词向量
        if ngrams_found == 0:
            for ngram in ngrams_single:
                if ngram in wv_from_text.index2word:
                    word_vec += wv_from_text[ngram]
                    ngrams_found += 1
        if word_vec.any():  # 只要有一个不为0
            return word_vec / max(1, ngrams_found)
        else:
            print('all ngrams for word %s absent from model' % word)
            return 0

In [ ]:
# 给予余弦相似度的相似度计算 加入idf 权重 k(word)  加到sum---
def similarity_cosine_weighted(wv_from_text, comment, summary):
    vector1 = np.zeros(200)
    jaccard_similarity(comment, summary)
    # 如果在词典之中，直接返回词向量
    for word in word_list1:
        vector1 += word_vec(word, wv_from_text)
    vector1 = vector1 / len(word_list1)
    # print(vector1)
    vector2 = np.zeros(200)
    for word in word_list2:
        vector2 += word_vec(word, wv_from_text)
    vector2 = vector2 / len(word_list2)
    # print(vector2)
    cos1 = np.sum(vector1 * vector2)
    cos21 = np.sqrt(sum(vector1 ** 2))
    cos22 = np.sqrt(sum(vector2 ** 2))
    similarity =  cos1 / float(cos21 * cos22)
    return similarity

In [102]:
def similarity_main(wv_from_text, comment, summary):
    # 测试句子相似度
    start_time = time.time()
    # vec_summary = wordVec(sentence_danmu1, wv_from_text, min_n=1, max_n=3)  # 词向量获取
    word_list1 = [word.word for word in pesg.cut(comment) if word.flag[0] not in ['w', 'x', 'u']]
    word_list2 = [word.word for word in pesg.cut(summary) if word.flag[0] not in ['w', 'x', 'u']]
    y = similarity_cosine(wv_from_text, word_list1, word_list2)
    #y = jaccard_similarity(word_list1, word_list2)
    #print("------vec_similarity:", y)
    #print("----loading time:", time.time() - start_time)
    return y

In [104]:
def get_similarity(clean_summary_dir, clean_comment_cut_dir, matrix_similarity_dir):
    # 加载comment
    pathDir = os.listdir(clean_summary_dir)

   
    for idx, file_name_txt in enumerate(pathDir):
        datas = []
        danmu_file = file_name_txt.split("_")
        danmu_comment = danmu_file[2]
        if int(danmu_comment) != 0:
            danmu_file_path = os.path.join(clean_comment_cut_dir, danmu_file[0])

            if os.path.exists(danmu_file_path):
                danmu = load_danmu(danmu_file_path)

                origin_path = os.path.join(clean_summary_dir, file_name_txt)
                summary = load_danmu(origin_path)

                # n个剧情简介对应M块弹幕
                similarity_list = []
                for n in summary:
                    ps_init = 0
                    s_score = []
                    danmu_cut = ""
                    for m in danmu:
                        if ps_init == m["cut"]:
                            danmu_cut += " " + m["danmaku"]
                        else:
                            # todo 2. 计算句子相似度
                            jjs = similarity_main(wv_from_text,danmu_cut, n["content"])
                            #jjs = similarity_cosine_weighted(wv_from_text, danmu_cut, n["content"])
                            print(jjs)
                            s_score.append(jjs)
                            ps_init += 1
                            danmu_cut = m["danmaku"]
                    similarity_list.append(s_score)

                # 保存相似度矩阵
                # matrix_similarity = os.path.join(matrix_similarity_dir, danmu_file)
                # save_similarity_cosine(similarity_list, matrix_similarity)
                #print(danmu_file[0])
                #datas.append({"danmu_file": danmu_file[0], "matrix_similarity": similarity_list})
                matrix_similarity = os.path.join(matrix_similarity_dir, danmu_file[0])
                save_similarity_cosine(similarity_list, matrix_similarity)
                # dump_to_json(datas, open(save_path, 'w', encoding='utf8'))
    print("---------finished---------")

In [ ]:
if __name__ == '__main__':
    # clean_comment_cut_dir = '../data/clean_comment_cut/'
    # clean_summary_dir = '../data/Entertainment_summary/'
    # clean_comment_cut_dir = 'E:\danmu-201909\data\clean_comment_cut_new/'

    clean_comment_cut_dir = '/home/baiqingchun/0000/danmaku/data/clean_comment_cut_new/'
    clean_summary_dir = '/home/baiqingchun/0000/danmaku/data/Entertainment_summary/'
    matrix_similarity_dir = '/home/baiqingchun/0000/danmaku/data/tfidf_matrix_similarity/'

    get_similarity(clean_summary_dir, clean_comment_cut_dir, matrix_similarity_dir)

0.0242424242424
0.0177777777778
0.0
0.0514285714286
0.00434782608696
0.00531914893617
0.0407239819005
0.0
0.0663900414938
0.0
0.0398230088496
0.0
0.0396475770925
0.0
0.0463917525773
0.0
0.0
0.0407239819005
0.0
0.0
0.038961038961
0.0
0.0
0.042654028436
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.00505050505051
0.0
0.0
0.0
0.00606060606061
0.0
0.0
0.00546448087432
0.0
0.0
0.00537634408602
0.0
0.00438596491228
0.0
0.00574712643678
0.0
0.00502512562814
0.00471698113208
0.0
0.0
0.00518134715026
0.00448430493274
0.0
0.00478468899522
0.00452488687783
0.0
0.00478468899522
0.00452488687783
0.0
0.00473933649289
0.00487804878049
0.0
0.00518134715026
0.0042194092827
0.00518134715026
0.00416666666667
0.00485436893204
0.142857142857
0.0379746835443
0.0888888888889
0.134408602151
0.03734439